<a href="https://colab.research.google.com/github/Adityadeeenair/Fake-News-Prediction-model/blob/main/Fake_News_Prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IMPORTING THE LIBRARIES**

In [13]:
import numpy as np
import pandas as pd
import re

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [14]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

**DATA PREPROCESSING :**

In [16]:
news = pd.read_csv("/content/train.csv")
news.title

,title
0,House Dem Aide: We Didn’t Even See Comey’s Let...
1,"FLYNN: Hillary Clinton, Big Woman on Campus - ..."
2,Why the Truth Might Get You Fired
3,15 Civilians Killed In Single US Airstrike Hav...
4,Iranian woman jailed for fictional unpublished...
...,...
20795,Rapper T.I.: Trump a ’Poster Child For White S...
20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -..."
20797,Macy’s Is Said to Receive Takeover Approach by...
20798,"NATO, Russia To Hold Parallel Exercises In Bal..."


In [17]:
news.shape

(20800, 5)

**CHECKING AND HANDLING MISSING VALUES :**

In [18]:
news.isnull().sum()

,0
id,0
title,558
author,1957
text,39
label,0


In [19]:
news = news.fillna(' ')

**MERGING AND MAKING A NEW COLUMN -> CONTENTS**


In [20]:
news['contents'] = news['author'] + ' ' + news['title']

In [21]:
print(news['contents'])

0        Darrell Lucus House Dem Aide: We Didn’t Even S...
1        Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...
2        Consortiumnews.com Why the Truth Might Get You...
3        Jessica Purkiss 15 Civilians Killed In Single ...
4        Howard Portnoy Iranian woman jailed for fictio...
                               ...                        
20795    Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...
20796    Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma...
20797    Michael J. de la Merced and Rachel Abrams Macy...
20798    Alex Ansary NATO, Russia To Hold Parallel Exer...
20799              David Swanson What Keeps the F-35 Alive
Name: contents, Length: 20800, dtype: object


**SEPARATING THE DATA AND THE LABEL :**

In [22]:
X = news.drop(columns = 'label' , axis = 1 )
Y = news['label']

In [23]:
print(X)

          id                                              title  \
0          0  House Dem Aide: We Didn’t Even See Comey’s Let...   
1          1  FLYNN: Hillary Clinton, Big Woman on Campus - ...   
2          2                  Why the Truth Might Get You Fired   
3          3  15 Civilians Killed In Single US Airstrike Hav...   
4          4  Iranian woman jailed for fictional unpublished...   
...      ...                                                ...   
20795  20795  Rapper T.I.: Trump a ’Poster Child For White S...   
20796  20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   
20797  20797  Macy’s Is Said to Receive Takeover Approach by...   
20798  20798  NATO, Russia To Hold Parallel Exercises In Bal...   
20799  20799                          What Keeps the F-35 Alive   

                                          author  \
0                                  Darrell Lucus   
1                                Daniel J. Flynn   
2                             Consortiu

In [24]:
print(Y)

0        1
1        0
2        1
3        1
4        1
        ..
20795    0
20796    0
20797    0
20798    1
20799    1
Name: label, Length: 20800, dtype: int64


**STEMMING** using **PorterStemmer**:

In [25]:
pstem = PorterStemmer()

In [26]:
def Stemmer(contents):
  Stemmed_data = re.sub('[^a-zA-Z]', ' ' , contents)
  Stemmed_data = Stemmed_data.lower()
  Stemmed_data = Stemmed_data.split()
  Stemmed_data = [pstem.stem(word) for word in Stemmed_data if not word in stopwords.words('english')]
  Stemmed_data = ' '.join(Stemmed_data)
  return Stemmed_data

In [27]:
news['contents'] = news['contents'].apply(Stemmer)

In [28]:
print(news['contents'])

0        darrel lucu hous dem aid even see comey letter...
1        daniel j flynn flynn hillari clinton big woman...
2                   consortiumnew com truth might get fire
3        jessica purkiss civilian kill singl us airstri...
4        howard portnoy iranian woman jail fiction unpu...
                               ...                        
20795    jerom hudson rapper trump poster child white s...
20796    benjamin hoffman n f l playoff schedul matchup...
20797    michael j de la merc rachel abram maci said re...
20798    alex ansari nato russia hold parallel exercis ...
20799                            david swanson keep f aliv
Name: contents, Length: 20800, dtype: object


**SEPARATING CONTENTS AND LABEL FROM THE DATASET :**

In [29]:
X = news['contents'].values
Y = news['label'].values

In [30]:
print(X)
print()
print(Y)

['darrel lucu hous dem aid even see comey letter jason chaffetz tweet'
 'daniel j flynn flynn hillari clinton big woman campu breitbart'
 'consortiumnew com truth might get fire' ...
 'michael j de la merc rachel abram maci said receiv takeov approach hudson bay new york time'
 'alex ansari nato russia hold parallel exercis balkan'
 'david swanson keep f aliv']

[1 0 1 ... 0 1 1]


In [31]:
print(X.shape, Y.shape)

(20800,) (20800,)


**CONVERTING TEXT DATA INTO NUMERIC DATA**

In [32]:
vect = TfidfVectorizer()
vect.fit(X)
X = vect.transform(X)

In [33]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 210687 stored elements and shape (20800, 17128)>
  Coords	Values
  (0, 267)	0.2701012497770876
  (0, 2483)	0.36765196867972083
  (0, 2959)	0.24684501285337127
  (0, 3600)	0.3598939188262558
  (0, 3792)	0.27053324808454915
  (0, 4973)	0.23331696690935097
  (0, 7005)	0.2187416908935914
  (0, 7692)	0.24785219520671598
  (0, 8630)	0.2921251408704368
  (0, 8909)	0.36359638063260746
  (0, 13473)	0.2565896679337956
  (0, 15686)	0.2848506356272864
  (1, 1497)	0.2939891562094648
  (1, 1894)	0.15521974226349364
  (1, 2223)	0.3827320386859759
  (1, 2813)	0.19094574062359204
  (1, 3568)	0.26373768806048464
  (1, 5503)	0.7143299355715573
  (1, 6816)	0.1904660198296849
  (1, 16799)	0.30071745655510157
  (2, 2943)	0.3179886800654691
  (2, 3103)	0.46097489583229645
  (2, 5389)	0.3866530551182615
  (2, 5968)	0.3474613386728292
  (2, 9620)	0.49351492943649944
  :	:
  (20797, 3643)	0.2115550061362374
  (20797, 7042)	0.21799048897828685
  (2079

**SPLITTING THE DATA INTO TRAINING AND TESTING DATA**

In [34]:
X_train , X_test , Y_train , Y_test = train_test_split(X, Y, test_size = 0.2, stratify = Y, random_state = 2)

**TRAINING THE LOGISTIC REGRESSION MODEL**

In [35]:
model = LogisticRegression()

In [36]:
model.fit(X_train, Y_train)  #Setting up the model

LogisticRegression()

In [37]:
import joblib

# Save artifacts
joblib.dump(model, "fake_news_model.joblib")
joblib.dump(vect, "tfidf_vectorizer.joblib")

print("Artifacts saved")


Artifacts saved


**EVALUATING THE MODEL**:


In [38]:
#Predicting the labels on training data
train_prediction = model.predict(X_train)

#Checking the accuracy score of predictions on training data
train_accuracy = accuracy_score(train_prediction, Y_train)

In [39]:
print("Accuracy of the training data :" , train_accuracy)

Accuracy of the training data : 0.9863581730769231


In [40]:
#Predicting the labels on testing data
test_prediction = model.predict(X_test)

#Checking the accuracy score of predictions on testing data
test_accuracy = accuracy_score(test_prediction, Y_test)

In [41]:
print("Accuracy of the testing data :" , test_accuracy)

Accuracy of the testing data : 0.9790865384615385


**MAKING A PREDICTIVE SYSTEM :**

In [43]:
input_news = "Avoiding Peanuts to Avoid an Allergy Is a Bad Strategy for Most The New York Times"


#preprocessing the input data
processed_text = Stemmer(input_news)

#converting text into numerical form using the trained vectorizer
vectorized_input = vect.transform([processed_text])

#Prediction
prediction = model.predict(vectorized_input)
print(prediction)

if prediction[0] == 0:
    print("The news is Real")
else:
    print("The news is Fake")

[0]
The news is Real
